<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
<a href="http://mng.bz/orYv">Build a Large Language Model From Scratch</a> 책의 보조 코드 by <a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>코드 저장소: <a href="https://github.com/rasbt/LLMs-from-scratch">https://github.com/rasbt/LLMs-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="http://mng.bz/orYv"><img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>

# 지시 데이터셋을 위한 "수동태" 항목 생성하기

- 이 노트북은 OpenAI의 GPT-4를 사용하여 지시 데이터셋을 위한 "수동태(passive voice)" 항목을 생성합니다. 다음은 예시입니다:

```python
{  
   'instruction': 'Identify the verb in the following sentence',
   'input': 'The cat sleeps on the couch.',
   'output': 'The verb in the sentence is "sleeps."',
   'output_2': 'The sentence is "sleeps."'   #  <---- 새로 생성된 항목
}  
```

In [ ]:
# pip install -r requirements-extra.txt

In [2]:
from importlib.metadata import version

pkgs = ["openai",  # OpenAI API
        "tqdm",    # 진행률 표시줄
       ]

for p in pkgs:
    print(f"{p} version: {version(p)}")

openai version: 1.30.3
tqdm version: 4.65.0


## OpenAI API 테스트

- 먼저 OpenAI API가 올바르게 설정되어 있는지 테스트해보겠습니다
- 아직 계정이 없다면 https://platform.openai.com/ 에서 계정을 만들어야 합니다
- GPT-4 API는 무료가 아니므로 계정에 일부 자금을 충전해야 합니다 (https://platform.openai.com/settings/organization/billing/overview 참조)
- 이 노트북의 코드를 사용하여 ~200개의 수동태 항목을 생성하는 데 약 $0.13 (13센트)가 소요됩니다

- 먼저 OpenAI API 비밀 키를 제공해야 합니다. 이는 https://platform.openai.com/api-keys 에서 찾을 수 있습니다
- 이 키를 다른 사람과 공유하지 않도록 주의하세요
- 이 비밀 키(`"sk-..."`)를 이 폴더의 `config.json` 파일에 추가하세요

In [3]:
import json
from openai import OpenAI

# JSON 파일에서 API 키를 로드합니다.
# "sk-..."를 https://platform.openai.com/api-keys의 실제 API 키로 바꾸세요
with open("config.json", "r") as config_file:
    config = json.load(config_file)
    api_key = config["OPENAI_API_KEY"]

client = OpenAI(api_key=api_key)

- 먼저 간단한 예제로 API를 시도해서 의도한 대로 작동하는지 확인해보겠습니다:

In [4]:
def run_chatgpt(prompt, client, model="gpt-4-turbo"):
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.0,
    )
    return response.choices[0].message.content


# 입력 준비
sentence = "I ate breakfast"
prompt = f"Convert the following sentence to passive voice: '{sentence}'"
run_chatgpt(prompt, client)

'Breakfast was eaten by me.'

## JSON 항목 생성

- 다음으로 수정하려는 파일을 로드합니다:

In [5]:
import json

json_file = "instruction-examples.json"

with open(json_file, "r") as file:
    json_data = json.load(file)
    
print("Number of entries:", len(json_data))

Number of entries: 200


- 먼저 작은 샘플에서 OpenAI 채팅 API를 시도해서 올바르게 작동하는지 확인해보겠습니다:

In [6]:
for entry in json_data[:5]:
    text = entry["output"]
    prompt = f"Without adding any response or explanation, convert the following text to passive voice: {text}"
    
    print("\nInput:")
    print(">>", text)
    print("\nOutput:")
    print(">>", run_chatgpt(prompt, client))
    print("\n-------------------------")


Input:
>> The verb in the sentence is "sleeps."

Output:
>> The sentence is "sleeps."

-------------------------

Input:
>> The plural form of "goose" is "geese."

Output:
>> The plural form of "goose" is referred to as "geese."

-------------------------

Input:
>> The three primary colors are red, blue, and yellow.

Output:
>> Red, blue, and yellow are considered the three primary colors.

-------------------------

Input:
>> They had finished the game.

Output:
>> The game had been finished by them.

-------------------------

Input:
>> The abbreviation for "Doctor of Philosophy" is Ph.D.

Output:
>> The abbreviation "Ph.D." is used for "Doctor of Philosophy".

-------------------------


- 이제 코드를 확장하여 생성된 항목을 `json_data`에 추가하고 진행률 표시줄을 추가해보겠습니다:

In [7]:
from tqdm import tqdm  # 진행률 표시줄 도구


for i, entry in tqdm(enumerate(json_data[:5]), total=len(json_data[:5])):
    text = entry["output"]
    prompt = f"Without adding any response or explanation, convert the following text to passive voice: {text}"
    json_data[i]["output_2"] = run_chatgpt(prompt, client)

100%|██████████████████████████████████████████████████████████████████████| 5/5 [00:04<00:00,  1.23it/s]


- 한 번 더, 새로운 항목들(`"output_2"`)이 괜찮은지 확인해보겠습니다

In [8]:
json_data[0]

{'instruction': 'Identify the verb in the following sentence: The cat sleeps on the couch.',
 'input': '',
 'output': 'The verb in the sentence is "sleeps."',
 'output_2': 'The sentence is "sleeps."'}

- 마지막으로, 위의 모든 것이 괜찮아 보이면 전체 json 데이터셋에 대해 수동태 변환을 실행해보겠습니다 (약 3분 소요):

In [9]:
for i, entry in tqdm(enumerate(json_data), total=len(json_data)):
    text = entry["output"]
    prompt = f"Without adding any response or explanation, convert the following text to passive voice: {text}"
    json_data[i]["output_2"] = run_chatgpt(prompt, client)

100%|██████████████████████████████████████████████████████████████████| 200/200 [03:43<00:00,  1.12s/it]


- 변환이 완료된 후 파일을 저장합니다:

In [10]:
new_json_file = json_file.replace(".json", "-modified.json")


with open(new_json_file, "w") as file:
    json.dump(json_data, file, indent=4)  # 예쁜 출력을 위한 "indent"